# SageMaker AI MLflow - RLAIF Fine-Tuning with AI Feedback

In this notebook, you will use **Reinforcement Learning from AI Feedback (RLAIF)** to fine-tune Qwen2.5-7B. Unlike SFT (which mimics good outputs) or DPO (which learns from preference pairs), RLAIF uses an **AI judge** to score model-generated responses during training — the model learns by trial and error with AI feedback.

**What you will learn:**
- Prepare training data in VERL post-training format from MLflow traces
- Create a medical triage reward prompt (AI judge scoring criteria)
- Register the reward prompt as an Evaluator in SageMaker AI Registry
- Run serverless RLAIF fine-tuning with RLAIFTrainer on Qwen2.5-7B Instruct
- Deploy and compare the RLAIF model with SFT/DPO models

### How RLAIF Works

| Step | What Happens |
|---|---|
| 1. Model generates | The base model generates responses to training prompts |
| 2. AI judge scores | An AI judge model scores each response using your reward prompt |
| 3. Model optimizes | The model is updated via RL to maximize the judge score |

### SFT vs DPO vs RLAIF

| Aspect | SFT | DPO | RLAIF |
|---|---|---|---|
| **Data** | prompt + completion | prompt + chosen + rejected | prompts only |
| **Feedback** | Static (from dataset) | Static (preference pairs) | Dynamic (AI judge scores during training) |
| **What it teaches** | What to say | What to prefer | How to maximize quality |

### Prerequisites
- Completed the agent tracing lab (04-1) with traces in MLflow
- A SageMaker AI MLflow App
- Service quota for serverless fine-tuning and ml.g5.2xlarge for deployment


<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>Warning:</strong> The cell below installs libraries and restarts the kernel. After the restart, continue with the next cell.
</div>

In [ ]:
# Install SageMaker SDK v3 and dependencies
%pip install "sagemaker>=3.5.0" mlflow==3.9.0 sagemaker-mlflow==0.2.0 --quiet

# restart kernel
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

## 1. Environment Setup

In [ ]:
import json
import os
import boto3
import mlflow
import pandas as pd
from sagemaker.core.helper.session_helper import Session, get_execution_role
from utils.trace_utils import extract_prompt_response

sess = Session()
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
region = sess.boto_region_name
s3_client = boto3.client("s3")

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

%store -r

print(f"Role: {role}")
print(f"Bucket: {bucket_name}")
print(f"Region: {region}")

## 2. Configure SageMaker AI MLflow

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

agent_experiment_name = "medical-triage-agent"
datacapture_experiment_name = "medical-triage-datacapture"
offline_eval_experiment_name = "medical-triage-offline-eval"

rlaif_experiment_name = "medical-triage-rlaif"
mlflow.set_experiment(rlaif_experiment_name)

print(f"MLflow tracking server: {mlflow.get_tracking_uri()}")
print(f"RLAIF experiment: {rlaif_experiment_name}")

## 3. Extract Prompts from MLflow Traces

For RLAIF, we only need **prompts** (not completions). During training, the model generates its own responses which are scored by the AI judge.
We pull prompts from all three experiments to build a diverse training set.

In [ ]:
# Load traces from all experiments
mlflow.set_experiment(agent_experiment_name)
agent_traces = mlflow.search_traces()
print(f"Traces in {agent_experiment_name}: {len(agent_traces)}")

mlflow.set_experiment(datacapture_experiment_name)
dc_traces = mlflow.search_traces()
print(f"Traces in {datacapture_experiment_name}: {len(dc_traces)}")

mlflow.set_experiment(offline_eval_experiment_name)
offline_traces = mlflow.search_traces()
print(f"Traces in {offline_eval_experiment_name}: {len(offline_traces)}")

all_traces = pd.concat([agent_traces, dc_traces, offline_traces], ignore_index=True)
print(f"\nTotal combined traces: {len(all_traces)}")

mlflow.set_experiment(rlaif_experiment_name)

In [ ]:
# Extract unique prompts from traces
prompts = set()

for idx, trace in all_traces.iterrows():
    prompt, response = extract_prompt_response(trace)
    if prompt and len(prompt.strip()) > 10:
        prompts.add(prompt.strip())

prompts = list(prompts)
print(f"Unique prompts extracted: {len(prompts)}")

# Preview
for i, p in enumerate(prompts[:3]):
    print(f"\nPrompt {i+1}: {p[:150]}...")

## 4. Prepare Training Data in VERL Format

RLAIF training requires data in the [VERL post-training format](https://verl.readthedocs.io/en/latest/preparation/prepare_data.html).
Each record contains the prompt in chat message format, a data source identifier, an ability tag, and reward model configuration.
The model generates its own responses during training - we only provide prompts.

In [ ]:
# Convert prompts to VERL format
verl_records = []

for prompt in prompts:
    verl_records.append({
        "data_source": "medical-triage-mlflow-traces",
        "prompt": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "ability": "medical-triage",
        "reward_model": {
            "ground_truth": "",
            "style": "llmj"
        }
    })

print(f"VERL training records: {len(verl_records)}")

# Preview a sample
from rich.pretty import pprint
pprint(verl_records[0])

In [ ]:
# Save and upload training data
project_prefix = "medical-triage-rlaif"

os.makedirs("./rlaif_data", exist_ok=True)

train_file = f"./rlaif_data/{project_prefix}_train.jsonl"
with open(train_file, "w") as f:
    for record in verl_records:
        f.write(json.dumps(record) + "\n")

if default_prefix:
    s3_prefix = f"{default_prefix}/{project_prefix}"
else:
    s3_prefix = project_prefix

train_s3_key = f"{s3_prefix}/{project_prefix}_train.jsonl"
s3_client.upload_file(train_file, bucket_name, train_s3_key)
train_data_uri = f"s3://{bucket_name}/{train_s3_key}"

import shutil
shutil.rmtree("./rlaif_data")

print(f"Training data uploaded: {train_data_uri}")

## 5. Register Dataset in SageMaker AI Registry

In [ ]:
from sagemaker.ai_registry.dataset import DataSet

training_dataset = DataSet.create(
    name=f"{project_prefix}-train",
    source=train_data_uri,
    sagemaker_session=sess,
    wait=True,
)

print(f"Training dataset ARN: {training_dataset.arn}")

## 6. Create Medical Triage Reward Prompt

The reward prompt defines the scoring criteria the AI judge uses to evaluate model-generated responses during training.
Our prompt instructs the judge to score responses based on medical triage quality: correct diagnosis, appropriate triage level, complete treatment protocol, and professional tone.

In [ ]:
reward_prompt = (
    "You are an expert medical triage evaluator assessing the quality of a "
    "medical triage response. Given the original patient query and an LLM-generated "
    "response, rate the quality of the triage assessment based on the following aspects:\n\n"
    "- Correct identification of the likely medical condition\n"
    "- Appropriate triage level assignment (Emergency, Urgent, Semi-urgent, Low)\n"
    "- Complete and clinically appropriate treatment protocol\n"
    "- Professional medical tone and structured format\n"
    "- Patient safety considerations\n\n"
    "Instructions:\n"
    "- Read the patient query carefully.\n"
    "- Evaluate whether the response provides a clinically sound triage assessment.\n"
    "- Check that the triage level matches the severity of symptoms.\n"
    "- Verify the treatment protocol is appropriate for the identified condition.\n"
    "- Assign a score from 0 to 1:\n"
    "   - 1 means excellent triage: correct condition, appropriate level, complete protocol.\n"
    "   - 0 means poor triage: wrong condition, inappropriate level, or dangerous recommendations.\n\n"
    "Output Format:\n"
    "Return a JSON object with EXACTLY the following structure (no extra text):\n"
    '{\n  "score": <numeric>,\n  "reasoning": "<brief explanation of strengths and weaknesses>"\n}\n\n'
    "Prompt: {{ prompt }}\n"
    "LLM-generated response: {{ response }}"
)

# Save reward prompt
reward_prompt_file = f"{project_prefix}_reward_prompt.txt"
with open(reward_prompt_file, "w") as f:
    f.write(reward_prompt)

print("Reward prompt created.")
print(f"\nPreview:\n{reward_prompt[:300]}...")

In [ ]:
# Upload reward prompt to S3
reward_s3_key = f"{s3_prefix}/{reward_prompt_file}"
s3_client.upload_file(reward_prompt_file, bucket_name, reward_s3_key)
reward_prompt_uri = f"s3://{bucket_name}/{reward_s3_key}"

os.remove(reward_prompt_file)

print(f"Reward prompt uploaded: {reward_prompt_uri}")

### Register Reward Prompt as Evaluator

In [ ]:
from sagemaker.ai_registry.evaluator import Evaluator
from sagemaker.ai_registry.air_constants import REWARD_PROMPT

reward_prompt_evaluator = Evaluator.create(
    name=f"{project_prefix}-reward-prompt",
    type=REWARD_PROMPT,
    source=reward_prompt_uri,
    wait=True,
)

print(f"Reward prompt ARN: {reward_prompt_evaluator.arn}")

## 7. RLAIF Fine-Tuning with RLAIFTrainer

The RLAIFTrainer orchestrates the reinforcement learning loop: the base model generates responses,
the AI judge scores them using the reward prompt, and the model is updated to maximize the score.

In [ ]:
base_model_id = "huggingface-llm-qwen2-5-7b-instruct"

if default_prefix:
    rlaif_output_path = f"s3://{bucket_name}/{default_prefix}/{base_model_id}-rlaif"
else:
    rlaif_output_path = f"s3://{bucket_name}/{base_model_id}-rlaif"

os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = f"https://mlflow.sagemaker.{region}.app.aws"

print(f"Base model: {base_model_id}")
print(f"Output path: {rlaif_output_path}")

In [ ]:
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

rlaif_mpg_name = f"{base_model_id}-medical-triage-rlaif-mpg"

try:
    mpg = ModelPackageGroup.get(model_package_group_name=rlaif_mpg_name)
    print(f"Model Package Group exists: {rlaif_mpg_name}")
except ClientError:
    mpg = ModelPackageGroup.create(
        model_package_group_name=rlaif_mpg_name,
        model_package_group_description="Medical triage RLAIF models",
    )
    print(f"Created Model Package Group: {rlaif_mpg_name}")

In [ ]:
from sagemaker.train.rlaif_trainer import RLAIFTrainer
from sagemaker.ai_registry.evaluator import Evaluator

reward_model_id = "openai.gpt-oss-120b-1:0"
reward_prompt_eval = Evaluator.get(name=f"{project_prefix}-reward-prompt")

trainer = RLAIFTrainer(
    model=base_model_id,
    model_package_group=rlaif_mpg_name,
    reward_model_id=reward_model_id,
    reward_prompt=reward_prompt_eval.arn,
    training_dataset=training_dataset,
    s3_output_path=rlaif_output_path,
    sagemaker_session=sess,
    role=role,
    accept_eula=True,
)

print(f"RLAIFTrainer configured.")
print(f"Reward model: {reward_model_id}")
print(f"Reward prompt: {reward_prompt_eval.arn}")

In [ ]:
# Set hyperparameters
trainer.hyperparameters.max_epochs = 1

print("Hyperparameters:")
for k, v in trainer.hyperparameters.to_dict().items():
    print(f"  {k}: {v}")

<div style="padding: 15px; background-color: #d1ecf1; border-left: 5px solid #17a2b8; color: #0c5460;">
<strong>Note:</strong> RLAIF training takes longer than SFT or DPO because the model generates responses and the AI judge scores them at each step. Expect 30-60 minutes depending on dataset size.
</div>

In [ ]:
training_job = trainer.train(wait=True)

RLAIF_TRAINING_JOB_NAME = training_job.training_job_name
print(f"RLAIF Training job completed: {RLAIF_TRAINING_JOB_NAME}")

## 8. Deploy the RLAIF Fine-Tuned Model

<div style="padding: 15px; background-color: #f8d7da; border-left: 5px solid #dc3545; color: #721c24;">
<strong>Important:</strong> You only have 2x ml.g5.2xlarge endpoint instances available. If the SFT or DPO endpoint is still running, you must delete it before deploying the RLAIF model. Run the cell below to check.
</div>

In [ ]:
sm_client = boto3.client("sagemaker")

for ep_name in ["workshop-qwen25-7b-sft", "workshop-qwen25-7b-dpo"]:
    try:
        response = sm_client.describe_endpoint(EndpointName=ep_name)
        status = response["EndpointStatus"]
        print(f"WARNING: Endpoint '{ep_name}' is {status}. Delete it before deploying RLAIF model.")
    except sm_client.exceptions.ResourceNotFound:
        print(f"Endpoint '{ep_name}' not found. OK.")
    except Exception as e:
        if "Could not find" in str(e) or "does not exist" in str(e):
            print(f"Endpoint '{ep_name}' not found. OK.")
        else:
            raise

In [ ]:
# Uncomment and run ONLY if endpoints need to be deleted
# from sagemaker.core.resources import Endpoint
# Endpoint.get(endpoint_name="workshop-qwen25-7b-sft").delete()
# Endpoint.get(endpoint_name="workshop-qwen25-7b-dpo").delete()
# print("Endpoints deleted.")

In [ ]:
from sagemaker.core.resources import ModelPackage, ModelPackageGroup
from sagemaker.core import s3

model_package_version = "1"

mpg = ModelPackageGroup.get(rlaif_mpg_name)
rlaif_model_package_arn = (
    f"{mpg.model_package_group_arn.replace('model-package-group', 'model-package', 1)}/{model_package_version}"
)
print(f"Model Package ARN: {rlaif_model_package_arn}")

model_package = ModelPackage.get(rlaif_model_package_arn)

merged_model_s3_uri = (
    s3.s3_path_join(
        model_package.inference_specification.containers[0].model_data_source.s3_data_source.s3_uri,
        "checkpoints",
        "hf_merged",
    ) + "/"
)
print(f"Merged model S3 URI: {merged_model_s3_uri}")

In [ ]:
rlaif_model_name = f"{base_model_id}-medical-triage-rlaif"
rlaif_endpoint_config_name = f"{base_model_id}-medical-triage-rlaif-config"
rlaif_endpoint_name = "workshop-qwen25-7b-rlaif"

instance_type = "ml.g5.2xlarge"
health_check_timeout = 700

CONTAINER_VERSION = "0.36.0-lmi18.0.0-cu128"
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/djl-inference:{CONTAINER_VERSION}"

env = {
    "HF_MODEL_ID": "/opt/ml/model",
    "OPTION_TRUST_REMOTE_CODE": "true",
    "OPTION_MODEL_LOADING_TIMEOUT": "3600",
    "OPTION_TENSOR_PARALLEL_DEGREE": "max",
    "SERVING_FAIL_FAST": "true",
    "OPTION_ROLLING_BATCH": "disable",
    "OPTION_ASYNC_MODE": "true",
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service",
    "OPTION_DTYPE": "bf16",
    "OPTION_QUANTIZE": "fp8",
    "OPTION_MAX_MODEL_LEN": json.dumps(1024 * 8),
}

print(f"Endpoint: {rlaif_endpoint_name}")
print(f"Instance: {instance_type}")

In [ ]:
from sagemaker.core.resources import Model, Endpoint, EndpointConfig
from sagemaker.core.shapes import ContainerDefinition, ModelDataSource, S3ModelDataSource, ProductionVariant

fine_tuned_model = Model.create(
    model_name=rlaif_model_name,
    primary_container=ContainerDefinition(
        image=inference_image,
        model_data_source=ModelDataSource(
            s3_data_source=S3ModelDataSource(
                s3_uri=merged_model_s3_uri,
                s3_data_type="S3Prefix",
                compression_type="None",
            )
        ),
        environment=env,
    ),
    execution_role_arn=role,
)
print(f"Model created: {rlaif_model_name}")

endpoint_config = EndpointConfig.create(
    endpoint_config_name=rlaif_endpoint_config_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=rlaif_model_name,
            instance_type=instance_type,
            initial_instance_count=1,
            model_data_download_timeout_in_seconds=health_check_timeout,
        )
    ],
)
print(f"EndpointConfig created: {rlaif_endpoint_config_name}")

endpoint = Endpoint.create(
    endpoint_name=rlaif_endpoint_name, endpoint_config_name=rlaif_endpoint_config_name
)
endpoint.wait_for_status("InService")
print(f"Endpoint {rlaif_endpoint_name} is InService!")

## 9. Test: Compare RLAIF Model vs Teacher Model

In [ ]:
import io

sagemaker_runtime = boto3.client("sagemaker-runtime")
TEACHER_ENDPOINT = "workshop-qwen3-8b"

class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0
    def __iter__(self):
        return self
    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()
            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]
            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise
            if "PayloadPart" not in chunk:
                continue
            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

def invoke_rlaif_model(prompt):
    request_body = {
        "messages": [{"role": "user", "content": [{"type": "text", "text": prompt}]}],
        "max_tokens": 1024,
        "temperature": 0.3,
        "top_p": 0.9,
        "stop": ["<|im_end|>"],
        "stream": True,
    }
    response = sagemaker_runtime.invoke_endpoint_with_response_stream(
        EndpointName=rlaif_endpoint_name,
        Body=json.dumps(request_body),
        ContentType="application/json",
    )
    generated_text = ""
    for line in LineIterator(response["Body"]):
        if line:
            line_str = line.decode("utf-8")
            if not line_str.strip() or line_str.strip() == "data: [DONE]":
                continue
            if line_str.startswith("data: "):
                line_str = line_str[6:]
            try:
                data = json.loads(line_str)
                if "choices" in data:
                    for choice in data["choices"]:
                        if "delta" in choice and "content" in choice["delta"]:
                            content = choice["delta"]["content"]
                            generated_text += content
                            print(content, end="", flush=True)
            except json.JSONDecodeError:
                pass
    print()
    return generated_text

def invoke_teacher_model(prompt):
    request_body = {
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1024,
        "temperature": 0.3,
    }
    response = sagemaker_runtime.invoke_endpoint(
        EndpointName=TEACHER_ENDPOINT,
        Body=json.dumps(request_body),
        ContentType="application/json",
    )
    result = json.loads(response["Body"].read().decode("utf-8"))
    return result["choices"][0]["message"]["content"]

print("Inference utilities ready.")

In [ ]:
test_prompt = (
    "Assess the following patient and provide the likely condition, "
    "triage level, and treatment protocol.\n\n"
    "Patient ID: PT-001\nAge: 62\nSymptoms: chest pain, shortness of breath, sweating\n"
    "Reported Severity: high"
)

print("=" * 60)
print("RLAIF MODEL RESPONSE (Qwen2.5-7B RLAIF-trained):")
print("=" * 60)
rlaif_response = invoke_rlaif_model(test_prompt)
print()

print("=" * 60)
print("TEACHER MODEL RESPONSE (Qwen3-8B base):")
print("=" * 60)
teacher_response = invoke_teacher_model(test_prompt)
print(teacher_response)

In [ ]:
# Test with a non-urgent case
test_prompt_2 = (
    "Assess the following patient and provide the likely condition, "
    "triage level, and treatment protocol.\n\n"
    "Patient ID: PT-007\nAge: 29\nSymptoms: sneezing, runny nose, itchy eyes\n"
    "Reported Severity: low"
)

print("=" * 60)
print("RLAIF MODEL RESPONSE (Qwen2.5-7B RLAIF-trained):")
print("=" * 60)
rlaif_response = invoke_rlaif_model(test_prompt_2)
print()

print("=" * 60)
print("TEACHER MODEL RESPONSE (Qwen3-8B base):")
print("=" * 60)
teacher_response = invoke_teacher_model(test_prompt_2)
print(teacher_response)

## 10. Cleanup

Uncomment and run the cells below to delete deployed resources.

In [ ]:
# Delete RLAIF endpoint
# Endpoint.get(endpoint_name=rlaif_endpoint_name).delete()
# print(f"Deleted Endpoint: {rlaif_endpoint_name}")

# Delete endpoint config
# EndpointConfig.get(endpoint_config_name=rlaif_endpoint_config_name).delete()
# print(f"Deleted EndpointConfig: {rlaif_endpoint_config_name}")

# Delete model
# Model.get(model_name=rlaif_model_name).delete()
# print(f"Deleted Model: {rlaif_model_name}")

In [ ]:
# Also delete the teacher model endpoint if no longer needed
# Endpoint.get(endpoint_name="workshop-qwen3-8b").delete()
# print("Deleted teacher endpoint: workshop-qwen3-8b")